# Guia Completo do Projeto
## Previsão diária e mensal de irradiância solar GHI com Machine Learning

Este notebook é o material mais detalhado do projeto. Ele foi escrito para servir como base de estudo, defesa e rastreabilidade técnica: aqui estão explicados o problema, os dados, os cálculos, os modelos, os arquivos Python responsáveis e onde os resultados são gravados.

O projeto transforma séries históricas de **Irradiância Global Horizontal (GHI)** em um problema supervisionado de previsão. Para cada localidade, o modelo recebe apenas informações disponíveis até o período atual `t` e estima o GHI do próximo período `t+1`.

No fluxo diário, `t+1` é o dia seguinte. No fluxo mensal, `t+1` é o mês seguinte.

Este notebook cobre:

- problema de pesquisa, objetivo e escopo;
- origem, período, granularidade e unidade dos dados;
- diferença entre média diária, média mensal e média móvel;
- limpeza, quantização e normalização;
- criação e alinhamento das features temporais;
- separação cronológica entre treino e teste;
- funcionamento dos modelos XGBoost, MLP, RNN e LSTM;
- como as previsões e métricas são calculadas;
- quais arquivos `.py` executam cada etapa;
- onde ficam os CSVs, modelos salvos, previsões e gráficos;
- workflow dos modelos experimentais DilatedRNN, DeepAR e DeepNPTS;
- limitações, decisões metodológicas e perguntas comuns de banca.

> **Ideia central:** os notebooks documentam e apresentam os resultados, mas os cálculos oficiais ficam nos arquivos Python do projeto. Os principais resultados oficiais são gerados por `treinar_todas_localidades.py`, com apoio dos módulos em `codigo_fonte/`.

---

### Sumário

1. Problema, objetivo e escopo  
2. Conceitos de GHI, unidade e granularidade  
3. Dados e localidades  
4. Pipeline completo  
5. Coleta, validação e proveniência  
6. Média diária, mensal e móvel  
7. Limpeza e resolução diária  
8. Quantização e normalização  
9. Features temporais  
10. Alinhamento com o alvo e prevenção de vazamento  
11. Base final de modelagem  
12. Divisão cronológica treino/teste  
13. Modelos oficiais e funcionamento interno  
13.8. Modelos avançados experimentais  
14. Métricas, cálculos e desnormalização  
15. Gráficos, artefatos e arquivos de saída  
16. Resultados atuais  
17. Mapa dos arquivos Python  
18. Como executar  
19. Limitações e possíveis melhorias  
20. Perguntas comuns de banca  
21. Roteiro de apresentação  
22. Resumo final


# 1. Problema, objetivo e escopo

## Problema

A irradiância solar varia ao longo do tempo por causa de estação do ano, latitude, nebulosidade, condições atmosféricas e características regionais. Essa variação dificulta o planejamento de aplicações que dependem da disponibilidade de radiação solar.

O projeto pergunta: usando somente o histórico recente e suavizado do próprio GHI, é possível estimar o GHI médio do próximo período em localidades associadas a fábricas de veículos elétricos?

## Objetivo geral

Desenvolver e avaliar um pipeline de Machine Learning capaz de prever:

```text
Fluxo diário:  GHI médio diário do dia seguinte
Fluxo mensal:  GHI médio mensal do mês seguinte
```

## Formulação supervisionada

O problema é tratado como uma **regressão de série temporal**. A série histórica é convertida em linhas supervisionadas, nas quais as entradas são lags e médias móveis do GHI, e o alvo é o próximo período.

Em notação simplificada para o fluxo diário:

```text
ŷ(t+1) = f(GHI(t), GHI(t-1), GHI(t-2), GHI(t-6), médias móveis até t)
```

No fluxo mensal, a lógica é a mesma, mas cada linha representa um mês:

```text
ŷ(mês+1) = f(GHI(mês), GHI(mês-1), GHI(mês-2), GHI(mês-5), médias móveis mensais)
```

## Escopo atual

- 10 localidades associadas a fábricas de veículos elétricos;
- dados oficiais NLR/NSRDB, produto `GOES Aggregated PSM v4`;
- período de 1º de janeiro de 2019 a 31 de dezembro de 2024;
- série diária gerada pela média das observações horárias de cada dia;
- fluxo mensal complementar gerado pela média dos dias dentro de cada mês civil;
- 4 modelos oficiais: XGBoost, MLP, RNN e LSTM;
- 3 modelos experimentais em rodada separada: DilatedRNN, DeepAR_exp e DeepNPTS_aprox;
- divisão cronológica 80% treino e 20% teste;
- métricas MAE, MSE, RMSE, R² e nRMSE;
- avaliação na escala normalizada e na escala física aproximada em `W/m²`.

O projeto prevê **irradiância solar**, não geração elétrica, consumo energético ou produção de uma fábrica. O GHI em `W/m²` representa potência solar por metro quadrado em uma superfície horizontal ideal no ponto geográfico consultado. Para estimar geração fotovoltaica real seriam necessários outros dados, como área dos painéis, eficiência, inclinação, temperatura e perdas do sistema.


# 2. O que é GHI?

**GHI** significa *Global Horizontal Irradiance*, ou Irradiância Global Horizontal. É a irradiância solar total recebida por uma superfície horizontal.

Conceitualmente, inclui:

```text
GHI = componente direta projetada no plano horizontal + componente difusa
```

## Unidade usada

Os dados da API são declarados em:

```text
W/m²
```

Essa unidade representa **potência por área em um instante ou intervalo médio**.

## Um cuidado importante

Neste projeto, o valor diário é a **média das observações horárias em W/m²**. Portanto:

- o resultado continua em `W/m²`;
- ele expressa a irradiância média do dia;
- ele não é uma soma de energia diária em `Wh/m²` ou `kWh/m²/dia`.

Se o objetivo fosse energia solar diária acumulada, seria necessário integrar a irradiância ao longo do tempo, e não simplesmente calcular sua média.


# 3. Dados e localidades

## Fonte

Os dados vêm do **NLR/NSRDB**:

- NLR: *National Laboratory of the Rockies*;
- NSRDB: *National Solar Radiation Database*;
- produto: `GOES Aggregated PSM v4`;
- variável solicitada: `ghi`;
- intervalo da API: 60 minutos;
- período oficial do projeto: 2019–2024.

## Localidades

| Localidade | País |
|---|---|
| BYD Camaçari | Brasil |
| Tesla Gigafactory Nevada | EUA |
| Tesla Gigafactory Texas | EUA |
| Hyundai Metaplant Georgia | EUA |
| Rivian Normal | EUA |
| Tesla Fremont Factory | EUA |
| Lucid AMP 1 Casa Grande | EUA |
| GM Factory Zero | EUA |
| Ford Rouge Electric Vehicle Center | EUA |
| BMW San Luis Potosí | México |

Cada localidade possui seu próprio CSV, sua própria preparação e seus próprios modelos. Os dados das dez localidades **não são misturados em um único treinamento**.

## Tamanho das bases

Cada CSV bruto validado contém:

```text
2.192 observações diárias
01/01/2019 a 31/12/2024
365 + 366 + 365 + 365 + 365 + 366 dias
```

Os anos bissextos de 2020 e 2024 estão incluídos.


# 4. Pipeline completo

```text
Cadastro auditável das localidades
            ↓
Coleta horária de GHI pela API NLR/NSRDB
            ↓
Cálculo de estatísticas horárias: média, sigma e COV = sigma/média
            ↓
Agregação das observações para média diária
            ↓
Validação de origem, unidade, cobertura e integridade
            ↓
Limpeza e padronização da série
            ↓
Quantização do GHI em 128 níveis
            ↓
Normalização dos níveis para [0, 1]
            ↓
Criação de 4 lags + 3 médias móveis
            ↓
Criação do alvo do dia seguinte
            ↓
Remoção das linhas sem histórico/alvo
            ↓
Divisão cronológica: 80% treino / 20% teste
            ↓
Treinamento independente de XGBoost, MLP, RNN e LSTM
            ↓
Rodada experimental opcional com DilatedRNN, DeepAR_exp e DeepNPTS_aprox
            ↓
Previsões no período de teste
            ↓
Desnormalização das previsões para W/m²
            ↓
MAE, MSE, RMSE, R² e nRMSE nas escalas normalizada e W/m²
            ↓
CSVs, modelos e gráficos nas duas escalas
            ↓
Comparação dos modelos nas 10 localidades
```

## Regra metodológica central

Em qualquer linha usada pelo modelo:

```text
features ≤ dia t
alvo      = dia t+1
```

O futuro nunca pode entrar nas features. A normalização e a quantização também são ajustadas somente com o trecho de treinamento.


# 5. Coleta, validação e proveniência

## Coleta

Para cada ano e localidade, a função `coletar_ghi_nrel` solicita à API:

```python
parameters=("ghi",)
time_step=60
leap_day=True
utc=False
```

Depois, as observações são agregadas por dia:

```python
daily = df_year[["ghi"]].resample("D").mean()
```

## Por que existem tantos metadados?

O CSV registra, além de `data` e `ghi`:

- nome, país, latitude e longitude da fábrica;
- endereço e fonte oficial da localidade;
- elemento OpenStreetMap e método de obtenção das coordenadas;
- fonte, produto, versão e endpoint da API;
- intervalo, agregação e unidade;
- ponto da grade NSRDB, identificação, fuso e elevação;
- data UTC da coleta.

Esses campos tornam a coleta **auditável e reproduzível**.

## Validações executadas

O script confirma:

- todas as colunas obrigatórias;
- fonte igual a `NLR/NSRDB`;
- produto e endpoint esperados;
- intervalo de 60 minutos e agregação `media_diaria`;
- unidade `W/m2`;
- cobertura diária completa de 2019 a 2024;
- ausência de datas inválidas ou duplicadas;
- GHI diário entre 0 e 500 W/m²;
- correspondência das coordenadas e da localidade;
- distância de até 5 km entre a fábrica e o ponto da grade NSRDB;
- presença no manifesto e igualdade do hash SHA-256.

Dados sintéticos não são usados como substituição quando a API falha.


# 6. Como funcionam as médias?

Esta seção evita uma confusão comum: média diária, média mensal e média móvel não são a mesma coisa e entram em etapas diferentes do projeto.

## 6.1 Média diária

A API fornece observações em intervalo de 60 minutos. Para cada data, o projeto calcula a média aritmética dos valores horários daquele dia:

```text
GHI_diário(d) = média(GHI_00h, GHI_01h, ..., GHI_23h)
```

No código, a agregação aparece em `codigo_fonte/preprocessamento.py`, dentro de `coletar_ghi_nrel` e `garantir_resolucao_diaria`:

```python
daily = df_year[["ghi"]].resample("D").mean()
```

Como é uma média de valores em `W/m²`, a unidade continua `W/m²`. O projeto não integra energia em `Wh/m²` ou `kWh/m²/dia`.

## 6.2 Média mensal

O fluxo mensal começa a partir da série diária já validada. Os dias de cada mês civil são agrupados por média:

```text
GHI_mensal(m) = média dos GHI_diários dentro do mês m
```

No código, isso fica em `garantir_resolucao_mensal`:

```python
mensal = df.set_index("data")[["ghi"]].resample("ME").mean()
```

Esse fluxo cria uma base menor, com 72 meses brutos entre 2019 e 2024. Depois das janelas de 12 meses e da retirada do último mês sem alvo futuro, restam 60 exemplos de modelagem por localidade.

## 6.3 Média móvel

A média móvel é uma feature calculada linha a linha, sempre olhando para trás. Ela não agrupa por mês civil; ela usa uma janela deslizante.

Exemplo diário:

```text
média móvel de 3 dias em 30/01
= média de 28/01, 29/01 e 30/01
```

No dia seguinte, a janela anda:

```text
média móvel de 3 dias em 31/01
= média de 29/01, 30/01 e 31/01
```

As médias móveis são calculadas sobre o GHI quantizado e normalizado. No fluxo diário, as janelas são de 3, 7 e 30 dias. No fluxo mensal, são de 3, 6 e 12 meses.

| Tipo | Cálculo | Onde entra |
|---|---|---|
| Média diária | média das observações horárias de um dia | base principal do fluxo diário |
| Média mensal | média dos dias de um mês civil | base principal do fluxo mensal |
| Média móvel | média de uma janela que termina em `t` | feature dos modelos |


# 7. Limpeza e resolução diária

A função `limpar_serie_ghi`:

1. detecta automaticamente a coluna de data;
2. detecta a coluna de GHI;
3. converte datas para `datetime`;
4. converte o GHI para número;
5. remove datas e valores inválidos;
6. remove GHI negativo, por não ter interpretação física neste contexto;
7. ordena cronologicamente;
8. remove datas duplicadas, mantendo o último registro;
9. garante frequência diária com `resample("D").mean()`.

## Por que agregar novamente se os CSVs já são diários?

A função também aceita CSV, Excel ou Parquet fornecidos pelo usuário. Uma entrada pode ser horária, subdiária ou possuir mais de uma observação por dia. A etapa garante que o restante do pipeline sempre receba o mesmo contrato:

```text
data       ghi
2019-01-01 valor diário
2019-01-02 valor diário
...
```

## Dias ausentes

O `resample` cria a grade diária, e dias sem nenhuma observação ficam ausentes e são removidos. Nos dez CSVs oficiais, uma validação anterior exige cobertura diária completa, portanto não há lacunas no período 2019–2024.


# 8. Quantização e normalização

## 8.1 Quantização em 128 níveis

O GHI contínuo é mapeado para inteiros de 0 a 127:

```text
0, 1, 2, ..., 127
```

Fórmula conceitual:

```text
q = arredondar((x - mínimo) / (máximo - mínimo) × 127)
```

Valores fora da faixa ajustada são limitados aos extremos.

### Objetivo

- representar o sinal em uma escala discreta controlada;
- reduzir pequenas variações do valor contínuo;
- manter uma representação comum para os quatro modelos.

### Consequência

A quantização perde parte da precisão original. Ela é uma decisão metodológica do projeto, não uma exigência de XGBoost ou MLP.

## 8.2 Normalização

Depois da quantização:

```text
ghi_normalizado = ghi_quantizado / 127
```

O resultado fica no intervalo `[0, 1]`. Isso é especialmente importante para o MLP, cujo treinamento é sensível à escala.

## 8.3 Proteção contra vazamento

O mínimo e o máximo da quantização são ajustados usando somente o trecho pertencente ao treinamento, incluindo seus alvos. O conjunto de teste é transformado com esses mesmos parâmetros.

Assim, estatísticas do futuro não são usadas para definir a escala do passado.


# 9. Features temporais

O modelo recebe sete colunas:

| Feature | Significado em relação ao alvo `t+1` |
|---|---|
| `ghi_t-1` | GHI normalizado do dia `t` |
| `ghi_t-2` | GHI normalizado do dia `t-1` |
| `ghi_t-3` | GHI normalizado do dia `t-2` |
| `ghi_t-7` | GHI normalizado do dia `t-6` |
| `ghi_media_movel_3d` | média normalizada dos dias `t-2` a `t` |
| `ghi_media_movel_7d` | média normalizada dos dias `t-6` a `t` |
| `ghi_media_movel_30d` | média normalizada dos dias `t-29` a `t` |

## Atenção ao nome dos lags

No código, a linha de data `t` prevê `t+1`. Por isso:

```python
dados["ghi_t-1"] = dados["ghi_normalizado"].shift(0)
```

`ghi_t-1` significa **um dia antes do alvo**, não um dia antes da data da linha.

Os deslocamentos usados são:

```python
ghi_t-1 → shift(0)
ghi_t-2 → shift(1)
ghi_t-3 → shift(2)
ghi_t-7 → shift(6)
```

## Por que usar esses horizontes?

- 1, 2 e 3 dias: persistência e comportamento recente;
- 7 dias: referência de uma semana;
- média de 3 dias: nível recente;
- média de 7 dias: suavização semanal;
- média de 30 dias: tendência mais lenta e sazonalidade aproximada.

As médias móveis são calculadas sobre o **GHI quantizado e normalizado**, não diretamente sobre o valor original em W/m².


# 10. Alinhamento entre features e alvo

Considere valores normalizados fictícios:

| Data | GHI normalizado |
|---|---:|
| 01/01 | 0,20 |
| 02/01 | 0,30 |
| 03/01 | 0,40 |
| 04/01 | 0,50 |

Para a linha de 03/01:

```text
data da linha = 03/01
ghi_t-1      = 0,40  (03/01)
ghi_t-2      = 0,30  (02/01)
ghi_t-3      = 0,20  (01/01)
média 3d     = (0,20 + 0,30 + 0,40) / 3 = 0,30
data_alvo    = 04/01
ghi_alvo     = 0,50
```

O código cria o alvo com:

```python
dados["data_alvo"] = dados["data"].shift(-1)
dados["ghi_alvo"] = dados["ghi_normalizado"].shift(-1)
```

## Por que não há vazamento temporal?

- as janelas terminam no dia `t`;
- o alvo está no dia `t+1`;
- a divisão treino/teste preserva a ordem;
- a escala é ajustada no treino;
- o projeto possui teste automatizado que verifica esse alinhamento.

Uma forma errada seria deslocar a série para o futuro antes de calcular a média, pois isso permitiria que o valor a prever entrasse nas entradas.


# 11. Base final de modelagem

Principais colunas do arquivo `*_features.csv`:

| Grupo | Colunas |
|---|---|
| Referência | `data`, `ghi` |
| Transformações | `ghi_quantizado`, `ghi_normalizado` |
| Entradas | `ghi_t-1`, `ghi_t-2`, `ghi_t-3`, `ghi_t-7` |
| Entradas | `ghi_media_movel_3d`, `ghi_media_movel_7d`, `ghi_media_movel_30d` |
| Alvo | `data_alvo`, `ghi_alvo` |
| Auditoria do alvo | `ghi_alvo_quantizado`, `ghi_alvo_original` |

## Por que a primeira linha é 30/01/2019?

A maior janela precisa de 30 observações:

```text
01/01 a 30/01 → primeiro histórico completo de 30 dias
```

Também é necessário existir o dia seguinte como alvo. Por isso a última linha de entrada é 30/12/2024, cujo alvo é 31/12/2024.

## Contagem

```text
Base diária original:             2.192 linhas
Perda inicial pela janela de 30d:    29 linhas
Perda final pela ausência de t+1:     1 linha
Base de modelagem:                2.162 linhas
```

Essa remoção é esperada e não representa perda acidental de dados.


# 12. Divisão cronológica treino/teste

Depois da criação das features:

```text
80% inicial → treino
20% final   → teste
```

Para cada localidade:

```text
Total:  2.162 exemplos
Treino: 1.729 exemplos
Teste:    433 exemplos
```

Períodos dos alvos:

```text
Treino: 31/01/2019 a 25/10/2023
Teste:  26/10/2023 a 31/12/2024
```

## Por que não embaralhar?

Em uma aplicação real, o modelo é treinado com o passado e utilizado no futuro. Embaralhar poderia colocar observações futuras no treino e observações passadas no teste, produzindo uma avaliação artificialmente otimista.

## O que o teste representa?

O conjunto final simula uma utilização posterior ao período usado para ajuste. As métricas mostram o desempenho em dados cronologicamente mais novos e não vistos durante o treinamento.


# 13. Modelos oficiais

Os quatro modelos oficiais do projeto são: **XGBoost, MLP, RNN e LSTM**. Todos recebem a mesma base de entrada e tentam resolver a mesma tarefa:

```text
usar informações disponíveis até o período t
                ↓
prever o GHI médio do próximo período t+1
```

Isso é importante porque a comparação fica justa. Se um modelo foi melhor que outro, a diferença veio do algoritmo, não de uma base diferente.

## 13.0 O que entra e o que sai dos modelos?

Depois do pré-processamento, cada linha da base vira um exemplo de aprendizado. Essa linha contém:

- valores recentes de GHI;
- médias móveis que resumem o comportamento recente;
- o alvo, que é o GHI do próximo dia ou do próximo mês.

No treino, o modelo vê as entradas e também vê a resposta correta. Ele ajusta seus parâmetros para aproximar a resposta correta. No teste, ele recebe só as entradas e precisa prever a resposta.

```text
Treino:
features conhecidas + resposta correta -> modelo aprende

Teste:
features conhecidas -> modelo prevê -> compara com resposta real
```

Em termos de variáveis:

```text
X_train = entradas dos 80% iniciais
y_train = respostas reais dos 80% iniciais
X_test  = entradas dos 20% finais
y_test  = respostas reais dos 20% finais, usadas só para avaliar
```

As previsões mostradas nos resultados são feitas sobre `X_test`, ou seja, sobre os **20% finais que não foram usados para treinar**.

## 13.0.1 Exemplo simples de uma linha de entrada

Imagine uma linha diária fictícia, já normalizada entre 0 e 1:

| Feature | Valor fictício | Leitura |
|---|---:|---|
| `ghi_t-1` | 0,72 | hoje teve GHI alto |
| `ghi_t-2` | 0,68 | ontem também foi alto |
| `ghi_t-3` | 0,65 | anteontem foi alto |
| `ghi_t-7` | 0,40 | há uma semana estava mais baixo |
| `ghi_media_movel_3d` | 0,68 | os últimos 3 dias estão altos |
| `ghi_media_movel_7d` | 0,58 | a semana está em nível médio-alto |
| `ghi_media_movel_30d` | 0,52 | o mês está em nível médio |

A resposta real dessa linha, usada no treino, poderia ser:

```text
ghi_alvo = 0,70
```

Em português claro: olhando para hoje e para os resumos recentes, o modelo deve aprender que o próximo dia ficou perto de `0,70` na escala normalizada. No conjunto de teste, o modelo recebe uma linha parecida, mas não recebe `ghi_alvo`; ele precisa estimar esse valor sozinho.

## 13.1 Features usadas no fluxo diário

No fluxo diário, cada linha usa sete informações:

```text
ghi_t-1
ghi_t-2
ghi_t-3
ghi_t-7
ghi_media_movel_3d
ghi_media_movel_7d
ghi_media_movel_30d
```

A leitura correta é em relação ao alvo. Se o alvo é amanhã, então:

| Feature | Interpretação simples |
|---|---|
| `ghi_t-1` | GHI de hoje, um período antes do alvo |
| `ghi_t-2` | GHI de ontem |
| `ghi_t-3` | GHI de dois períodos atrás |
| `ghi_t-7` | GHI de seis períodos atrás, referência semanal no diário |
| `ghi_media_movel_3d` | média curta, captura comportamento muito recente |
| `ghi_media_movel_7d` | média semanal, suaviza oscilações de poucos dias |
| `ghi_media_movel_30d` | média mais longa, aproxima tendência mensal |

## 13.2 Features usadas no fluxo mensal

No fluxo mensal, a ideia é a mesma, mas a unidade de tempo muda. Cada linha representa um mês e o alvo é o mês seguinte.

```text
ghi_t-1
ghi_t-2
ghi_t-3
ghi_t-6
ghi_media_movel_3m
ghi_media_movel_6m
ghi_media_movel_12m
```

A média móvel de 12 meses é especialmente importante porque resume um ciclo anual, útil em séries solares com sazonalidade.

O arquivo que cria essas colunas é:

```text
codigo_fonte/features.py
```

O arquivo que treina os modelos é:

```text
codigo_fonte/modelos.py
```

Os scripts que chamam tudo isso são:

```text
treinamento_principal.py         # uma única série
treinar_todas_localidades.py     # dez localidades
```

---

# 13.3 XGBoost

## Ideia simples

O XGBoost é um conjunto de muitas árvores de decisão. Uma árvore de decisão funciona como uma sequência de perguntas.

Exemplo intuitivo:

```text
A média dos últimos 30 dias está alta?
O GHI de hoje está acima do normal?
A média semanal caiu?
```

Cada pergunta separa os casos em grupos. No fim, a árvore chega a uma previsão. O XGBoost não usa uma árvore só; ele usa várias árvores, uma tentando corrigir os erros da anterior.

## Exemplo aplicado à linha fictícia

Usando a linha fictícia anterior, uma árvore de decisão poderia seguir uma lógica como:

```text
A média móvel de 3 dias é maior que 0,60?
    sim -> o período recente está forte

O valor de hoje (`ghi_t-1`) é maior que 0,70?
    sim -> hoje também foi forte

A média de 30 dias é menor que a média de 3 dias?
    sim -> existe alta recente acima da tendência mensal

Previsão dessa árvore: próximo GHI tende a continuar alto
```

Essa árvore sozinha pode errar. O XGBoost cria várias árvores: uma árvore pode capturar continuidade de dias ensolarados, outra pode corrigir casos em que a semana estava caindo, outra pode corrigir localidades com maior variação sazonal.

## Como ele aprende?

O processo é sequencial:

1. o modelo começa com uma previsão inicial;
2. calcula os erros dessa previsão;
3. cria uma árvore para explicar parte desses erros;
4. adiciona essa árvore ao conjunto;
5. repete o processo várias vezes;
6. a previsão final combina todas as árvores.

Em forma resumida:

```text
previsão inicial
      ↓
árvore 1 corrige erros
      ↓
árvore 2 corrige erros restantes
      ↓
árvore 3 corrige novos erros
      ↓
previsão final = soma das correções
```

## Por que ele faz sentido neste projeto?

O XGBoost costuma funcionar muito bem em bases tabulares, como esta, em que cada linha tem colunas prontas de entrada. Ele consegue capturar relações não lineares, por exemplo:

```text
quando a média de 30 dias está alta, mas os últimos 3 dias caíram,
o próximo valor pode se comportar de forma diferente.
```

## Configuração usada

No arquivo `codigo_fonte/modelos.py`, a função `treinar_xgboost` usa:

```python
XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)
```

| Parâmetro | Explicação fácil |
|---|---|
| `n_estimators=300` | cria até 300 árvores no conjunto |
| `max_depth=3` | cada árvore é pequena, para evitar decorar demais o treino |
| `learning_rate=0.05` | cada árvore corrige pouco, deixando o aprendizado gradual |
| `subsample=0.9` | cada árvore usa 90% das linhas, o que ajuda a regularizar |
| `colsample_bytree=0.9` | cada árvore usa 90% das colunas |
| `objective="reg:squarederror"` | otimiza erro quadrático, adequado para regressão |
| `random_state=42` | fixa a aleatoriedade para facilitar reprodução |

## Como interpretar o XGBoost no trabalho?

Ele é o modelo tabular forte do projeto. Se ele vence, isso indica que as relações entre lags, médias móveis e alvo foram bem capturadas por divisões tipo árvore. Se ele perde para uma rede, isso sugere que a rede encontrou alguma combinação não linear mais adequada naquela localidade.

---

# 13.4 MLP

## Ideia simples

A MLP é uma rede neural tradicional, também chamada de rede totalmente conectada. Ela recebe as sete features de entrada e passa esses valores por camadas de neurônios artificiais.

Um neurônio faz basicamente isto:

```text
multiplica cada entrada por um peso
soma tudo
aplica uma função de ativação
entrega um novo valor para a próxima camada
```

A rede aprende ajustando os pesos.

## Arquitetura usada

No projeto, a MLP tem esta estrutura:

```text
7 entradas
   ↓
64 neurônios
   ↓
32 neurônios
   ↓
1 saída: previsão do GHI normalizado
```

No código:

```python
MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=42,
    learning_rate_init=0.001,
)
```

## O que cada parte significa?

| Item | Explicação fácil |
|---|---|
| `hidden_layer_sizes=(64, 32)` | duas camadas ocultas: a primeira maior, a segunda menor |
| `activation="relu"` | função que permite aprender relações não lineares |
| `solver="adam"` | algoritmo que ajusta os pesos automaticamente |
| `max_iter=1000` | limite de iterações de treinamento |
| `learning_rate_init=0.001` | tamanho inicial dos ajustes nos pesos |

## Exemplo aplicado à linha fictícia

Para a MLP, a mesma linha fictícia entra como um vetor numérico:

```text
[0,72, 0,68, 0,65, 0,40, 0,68, 0,58, 0,52]
```

A rede não faz perguntas explícitas como uma árvore. Ela combina os valores com pesos. Um neurônio da primeira camada poderia, por exemplo, aprender uma ideia parecida com:

```text
0,4 × ghi_t-1
+ 0,3 × ghi_media_movel_3d
+ 0,2 × ghi_media_movel_7d
- 0,1 × ghi_t-7
```

Isso não é uma regra escrita manualmente; é uma intuição de como os pesos podem combinar as entradas. Durante o treinamento, a rede ajusta esses pesos para que a saída fique perto do alvo real.

Se a saída inicial da MLP fosse `0,62` e o alvo real fosse `0,70`, o erro indicaria que a rede precisa aumentar a previsão para padrões parecidos com aquele.

## Como ela aprende?

Durante o treino, a MLP faz uma previsão, compara com o valor real e ajusta os pesos para reduzir o erro. Esse ciclo se repete muitas vezes.

```text
entrada -> previsão -> erro -> ajuste dos pesos -> nova previsão
```

## Por que a normalização é importante?

Redes neurais são sensíveis à escala dos dados. Como o projeto transforma o GHI para `[0, 1]`, a MLP treina de forma mais estável. Sem normalização, valores em `W/m²` poderiam dificultar a convergência.

## Como interpretar a MLP no trabalho?

A MLP tenta aprender combinações suaves e não lineares entre os lags e as médias móveis. Ela não enxerga uma sequência temporal explicitamente; para ela, as sete features são sete colunas de uma tabela.

---

# 13.5 RNN

## Ideia simples

A RNN é uma rede recorrente. Diferente da MLP, ela foi criada para ler dados em sequência. Em uma série temporal tradicional, a RNN leria algo como:

```text
valor no tempo 1 -> valor no tempo 2 -> valor no tempo 3 -> previsão
```

Neste projeto, os dados já estão resumidos em sete features. Então o código reinterpreta essas sete colunas como uma sequência curta.

```text
7 features tabulares -> sequência com 7 passos
```

## Como a entrada é transformada?

A matriz original tem este formato:

```text
(amostras, 7 features)
```

Para a RNN, ela vira:

```text
(amostras, 7 passos, 1 variável por passo)
```

No código, isso acontece no método `_reshape` da classe `KerasSequenceRegressor`:

```python
valores.reshape((valores.shape[0], valores.shape[1], 1))
```

## Arquitetura usada

```text
7 passos sequenciais
        ↓
SimpleRNN com 32 unidades
        ↓
camada densa com 16 neurônios
        ↓
saída com 1 valor entre 0 e 1
```

No código, a camada principal é:

```python
tf.keras.layers.SimpleRNN(32, activation="tanh")
```

A saída final usa `sigmoid`, que tende a produzir valores entre 0 e 1:

```python
tf.keras.layers.Dense(1, activation="sigmoid")
```

## Exemplo aplicado à linha fictícia

A RNN recebe os mesmos valores, mas em formato sequencial. Em vez de enxergar tudo de uma vez como uma tabela, ela lê passo a passo:

```text
passo 1: ghi_t-1 = 0,72
passo 2: ghi_t-2 = 0,68
passo 3: ghi_t-3 = 0,65
passo 4: ghi_t-7 = 0,40
passo 5: média móvel 3d = 0,68
passo 6: média móvel 7d = 0,58
passo 7: média móvel 30d = 0,52
```

A cada passo, a RNN atualiza uma memória interna. Depois de ler `0,72`, `0,68` e `0,65`, ela pode formar a ideia de que os dias recentes foram fortes. Quando lê `0,40` no passo semanal, percebe que o valor de uma semana atrás era menor. Ao ler as médias móveis, ela refina essa memória antes de gerar a previsão.

## O que a RNN tenta aprender?

Ela tenta interpretar a ordem das features. Por exemplo, ela recebe primeiro uma informação, depois outra, e vai atualizando um estado interno. Esse estado funciona como uma memória curta do que já foi lido.

```text
passo 1 -> memória
passo 2 -> memória atualizada
passo 3 -> memória atualizada
...
saída final -> previsão
```

## Cuidado importante

A RNN aqui não está lendo os 2.192 dias um por um. Ela lê uma sequência curta formada pelas sete features de cada exemplo. Portanto, ela é uma comparação recorrente dentro da base supervisionada, não uma modelagem direta da série bruta inteira.

## Como interpretar a RNN no trabalho?

Se a RNN vai bem, isso sugere que a forma sequencial das features ajudou o modelo a combinar lags e médias móveis. Se ela não melhora, pode ser porque a sequência é curta demais para explorar todo o potencial de uma rede recorrente.

---

# 13.6 LSTM

## Ideia simples

A LSTM também é uma rede recorrente, mas mais sofisticada que a RNN simples. Ela foi criada para lidar melhor com memória: decidir o que deve ser lembrado, atualizado ou esquecido.

A sigla LSTM significa **Long Short-Term Memory**, ou memória de curto e longo prazo.

## Diferença entre RNN e LSTM

Uma RNN simples atualiza sua memória a cada passo, mas pode ter dificuldade em manter informações relevantes. A LSTM adiciona mecanismos chamados **portas**.

De forma intuitiva:

```text
porta de esquecimento -> decide o que descartar
porta de entrada      -> decide o que guardar
porta de saída        -> decide o que entregar como informação final
```

## Arquitetura usada

A entrada é igual à da RNN:

```text
(amostras, 7 passos, 1 variável)
```

A arquitetura é:

```text
7 passos sequenciais
        ↓
LSTM com 32 unidades
        ↓
camada densa com 16 neurônios
        ↓
saída com 1 valor entre 0 e 1
```

No código:

```python
tf.keras.layers.LSTM(32, activation="tanh")
```

## Exemplo aplicado à linha fictícia

Na mesma sequência, a LSTM também lê os sete passos. A diferença é que ela possui mecanismos internos para decidir o que guardar e o que reduzir de importância.

Exemplo intuitivo:

```text
passos recentes: 0,72, 0,68, 0,65
    -> guardar informação de sequência recente alta

valor semanal: 0,40
    -> reconhecer diferença em relação à semana anterior

médias móveis: 0,68, 0,58, 0,52
    -> confirmar que a alta recente está acima da tendência mensal
```

A LSTM pode decidir que a informação dos últimos 3 dias deve ter mais peso que o valor de 7 períodos atrás. Essa decisão não é escrita manualmente; ela é aprendida durante o treino.

## Por que testar LSTM?

A LSTM é muito usada em problemas de série temporal. Mesmo que a sequência deste projeto seja curta, ela serve como comparação com uma recorrência mais poderosa que a RNN simples.

## Cuidado importante

Como a entrada tem apenas sete passos, a LSTM pode ser mais complexa do que o necessário. Ela não recebe uma sequência longa de dias ou meses; recebe uma sequência curta de features já construídas. Por isso, ela deve ser interpretada como um teste de arquitetura, não como uma garantia de melhor desempenho.

---

# 13.7 Como o treinamento vira previsão?

Para cada localidade e para cada frequência, o pipeline executa a mesma sequência:

```text
1. cria features e alvo
2. separa 80% treino e 20% teste
3. treina XGBoost, MLP, RNN e LSTM com os 80%
4. usa os modelos treinados para prever os 20% finais
5. compara previsão com valor real dos 20% finais
6. salva métricas, previsões, modelos e gráficos
```

Em pseudocódigo:

```python
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
y_pred = y_pred.clip(0, 1)
```

O `clip(0, 1)` garante que nenhuma previsão normalizada fique fora do intervalo esperado.

Depois disso, o projeto calcula métricas na escala normalizada e também converte a previsão para uma aproximação em `W/m²`.

## Onde os modelos treinados ficam salvos?

```text
resultados/modelos/localidades/*.joblib        # XGBoost e MLP diários
resultados/modelos/localidades/*.keras         # RNN e LSTM diários
resultados/modelos/localidades_mensal/*.joblib # XGBoost e MLP mensais
resultados/modelos/localidades_mensal/*.keras  # RNN e LSTM mensais
```

## Resumo comparativo fácil

| Modelo | Como pensar nele | O que ele tenta fazer | Principal cuidado |
|---|---|---|---|
| XGBoost | várias árvores corrigindo erros | separar padrões por regras e combinar muitas correções | é tabular, não sequencial |
| MLP | rede de neurônios conectados | aprender combinações não lineares das sete features | depende bastante da escala |
| RNN | rede que lê passos em ordem | tratar as sete features como sequência curta | não lê a série bruta completa |
| LSTM | RNN com memória mais controlada | guardar/esquecer informações entre passos | pode ser complexa para só sete passos |

## Exemplo final comparando os quatro modelos

Para a mesma linha fictícia:

```text
ghi_t-1 = 0,72
média 3d = 0,68
média 30d = 0,52
alvo real no teste = 0,70
```

Cada modelo chega à previsão por um caminho diferente:

| Modelo | Como ele raciocina no exemplo | Exemplo de previsão fictícia |
|---|---|---:|
| XGBoost | aplica regras em várias árvores e soma correções | 0,69 |
| MLP | combina todas as features por pesos aprendidos | 0,71 |
| RNN | lê as features em sequência e atualiza uma memória curta | 0,68 |
| LSTM | lê a sequência e decide o que guardar/esquecer internamente | 0,70 |

Depois, o pipeline compara cada previsão com o alvo real `0,70`. Se a previsão for `0,69`, o erro absoluto normalizado é `0,01`. Se for `0,62`, o erro é `0,08`. Esses erros entram nas métricas MAE, MSE, RMSE, R² e nRMSE.

Esse exemplo não é uma linha real do CSV; ele serve para entender a lógica dos modelos.

## Resumo em uma frase para apresentação

> “Todos os modelos recebem os mesmos lags e médias móveis. O XGBoost usa árvores de decisão em sequência; a MLP aprende combinações não lineares com camadas densas; a RNN lê as features como uma sequência curta; e a LSTM faz o mesmo com uma memória interna mais sofisticada.”


# 13.8 Modelos avançados experimentais

Além dos quatro modelos oficiais, foi criada uma rodada complementar para testar arquiteturas mais avançadas de série temporal. Esses testes ficam isolados no script `experimentos_redes_avancadas.py` e salvam resultados em `resultados/experimentos_redes_avancadas/`. Eles não alteram os CSVs oficiais nem substituem automaticamente XGBoost, MLP, RNN e LSTM.

## Workflow da rodada experimental

```text
CSVs oficiais das 10 localidades
            ↓
Mesma preparação do pipeline principal
            ↓
Fluxo diário ou mensal
            ↓
Mesmas features temporais e mesmo alvo t+1
            ↓
Divisão cronológica 80% treino / 20% teste
            ↓
Treino separado de DilatedRNN, DeepAR_exp e DeepNPTS_aprox
            ↓
Previsões no conjunto de teste
            ↓
Desnormalização para W/m²
            ↓
MAE, RMSE, R² e nRMSE nas mesmas escalas
            ↓
Comparação com XGBoost, MLP, RNN e LSTM
```

A regra temporal é a mesma do pipeline oficial: as features usam informações disponíveis até o período `t`, e o alvo é o próximo período. No fluxo diário, o próximo período é o dia seguinte; no fluxo mensal, é o mês seguinte.

## DilatedRNN

A `DilatedRNN` mantém a ideia de entrada sequencial usada por RNN e LSTM, mas cria leituras em diferentes escalas. As sete features são reformatadas como uma sequência curta com uma variável por passo. Em seguida, a rede cria ramos com subamostragem por fatores 1, 2 e 4.

```text
7 features → sequência 7 × 1
            ↓
ramos dilatados: fator 1, fator 2 e fator 4
            ↓
SimpleRNN em cada ramo
            ↓
concatenação dos ramos
            ↓
camada densa
            ↓
saída sigmoid em [0, 1]
```

A motivação é permitir que a rede observe padrões recentes e padrões mais espaçados sem mudar as features originais do projeto.

## DeepAR_exp

O `DeepAR_exp` é uma aproximação experimental inspirada no DeepAR. Ele usa uma LSTM para produzir dois valores: uma média prevista e uma dispersão estimada. Durante o treino, a função de perda é uma verossimilhança Gaussiana negativa.

```text
7 features → sequência 7 × 1
            ↓
LSTM
            ↓
camada densa
            ↓
saída com média e sigma
            ↓
perda Gaussiana
```

Na etapa de previsão, o valor usado como previsão final é a média estimada, limitada à escala normalizada. Como esta implementação é uma aproximação simplificada, ela deve ser tratada como experimento, não como uma implementação canônica do DeepAR.

## DeepNPTS_aprox

O `DeepNPTS_aprox` é um baseline não paramétrico inspirado em NPTS. Ele não treina pesos de uma rede neural. Em vez disso, guarda os exemplos de treino e, para cada exemplo de teste, procura históricos parecidos.

```text
linha de teste
            ↓
cálculo de distância para exemplos de treino
            ↓
seleção dos k vizinhos mais próximos
            ↓
pesos por similaridade e recência
            ↓
média ponderada dos alvos históricos
            ↓
previsão final
```

A ideia é responder: se o padrão atual de lags e médias móveis se parece com padrões antigos, qual foi o próximo valor observado nesses casos?

## Como os resultados são apresentados

O notebook `03_experimentos_redes_avancadas.ipynb` apresenta os resultados desses modelos. Ele mostra:

- status das execuções;
- média das métricas por modelo e frequência;
- resultados por localidade;
- melhor modelo experimental por localidade;
- comparação completa com os modelos oficiais;
- contagem de vitórias por modelo.

O critério de comparação principal é o maior `R2_wm2`, mantendo a leitura na escala física em `W/m²`.

# 14. Métricas, cálculos e desnormalização

As métricas oficiais são calculadas em `codigo_fonte/avaliacao.py`, principalmente nas funções `calcular_metricas`, `desnormalizar_ghi`, `salvar_metricas` e `salvar_previsoes`.

O modelo é treinado com o alvo quantizado e normalizado em `[0, 1]`. Depois da previsão, o pipeline calcula métricas em duas escalas:

```text
1. escala normalizada: compara y_test e y_pred em [0, 1]
2. escala W/m²: compara GHI real e previsão desnormalizada
```

## 14.1 Como a previsão volta para W/m²

A função `desnormalizar_ghi` usa os limites de GHI ajustados no treino:

```text
GHI_previsto_wm2 = y_pred_normalizado × (max_treino - min_treino) + min_treino
```

No código:

```python
valores.clip(0, 1) * (maximo - minimo) + minimo
```

Essa conversão é aproximada porque antes da normalização houve quantização em 128 níveis. Mesmo assim, ela permite interpretar o erro na unidade física do projeto.

## 14.2 MAE: erro absoluto médio

```text
MAE = média(|real - previsto|)
```

Interpretação:

- mede o erro médio sem elevar ao quadrado;
- na escala `W/m²`, pode ser lido diretamente como desvio médio físico;
- menor é melhor.

## 14.3 MSE: erro quadrático médio

```text
MSE = média((real - previsto)^2)
```

Interpretação:

- erros grandes pesam mais;
- é útil para o cálculo do RMSE;
- menor é melhor.

## 14.4 RMSE: raiz do erro quadrático médio

```text
RMSE = sqrt(MSE)
```

Interpretação:

- retorna à mesma unidade da variável avaliada;
- na escala normalizada, fica na escala `[0, 1]`;
- na escala desnormalizada, fica em `W/m²`;
- menor é melhor.

## 14.5 nRMSE: RMSE dividido pela média real

```text
nRMSE = RMSE / média(real)
nRMSE_% = 100 × nRMSE
```

Essa métrica ajuda a comparar localidades com médias de GHI diferentes. Um erro de 40 `W/m²` pesa de forma diferente em uma localidade com média baixa e em outra com média alta.

## 14.6 R²: coeficiente de determinação

```text
R² = 1 - soma((real - previsto)^2) / soma((real - média(real))^2)
```

Interpretação:

| Valor de R² | Leitura |
|---:|---|
| próximo de 1 | o modelo explica grande parte da variação observada |
| próximo de 0 | desempenho parecido com prever a média |
| menor que 0 | pior que usar a média como referência |

Nos resumos do projeto, o melhor modelo de cada localidade é escolhido pelo maior `R2_wm2`, pois essa métrica está alinhada à escala física desnormalizada.

## 14.7 COV horário

O COV horário não é uma métrica de erro do modelo. Ele descreve variabilidade dos dados horários antes da agregação diária:

```text
COV = sigma(GHI horário) / média(GHI horário)
COV_% = 100 × COV
```

A API fornece observações de 60 minutos, mas os CSVs salvos em `dados/brutos/localidades_ev/*.csv` já estão agregados para uma linha diária. Por isso, quando só esses CSVs diários estão disponíveis, o COV horário aparece como `indisponivel_csv_diario`.

Com os CSVs atuais seria possível calcular um COV diário, mas isso responderia outra pergunta: variação entre dias, não variação horária original.

## 14.8 Ordem dos cálculos dentro do pipeline

```text
1. modelo prevê y_pred normalizado
2. previsão é limitada para [0, 1]
3. métricas normalizadas são calculadas
4. y_test e y_pred são convertidos para W/m²
5. métricas em W/m² são calculadas
6. CSVs de métricas e previsões são salvos
```

Os resultados aparecem nas colunas:

```text
MAE_normalizado, RMSE_normalizado, R2_normalizado, nRMSE_percentual_normalizado
MAE_wm2, RMSE_wm2, R2_wm2, nRMSE_percentual_wm2
```


# 15. Gráficos, artefatos e arquivos de saída

O projeto salva resultados em arquivos para que as tabelas e notebooks possam ser auditados sem repetir o treino toda vez.

## 15.1 Gráficos por localidade

Para cada localidade são geradas duas versões dos gráficos: uma na escala normalizada e outra na escala desnormalizada em `W/m²`.

A função central é `salvar_graficos` em `codigo_fonte/graficos.py`. Ela cria:

1. série temporal de teste com real e modelos treinados;
2. real versus previsto de cada modelo;
3. dispersão real versus previsto de cada modelo.

Como interpretar:

- linhas próximas no gráfico temporal indicam bom acompanhamento da dinâmica;
- na dispersão, pontos próximos da diagonal representam previsões próximas do real;
- afastamentos sistemáticos podem indicar viés;
- dificuldade em picos e quedas mostra limitação diante de mudanças rápidas;
- a versão em `W/m²` mostra a magnitude física do erro.

## 15.2 Arquivos de features

As bases finais de modelagem ficam em:

```text
dados/processados/localidades_ev/*_features.csv
```

No fluxo mensal:

```text
dados/processados/localidades_ev/*_features_mensal.csv
```

Esses arquivos contêm `data`, `ghi`, `ghi_quantizado`, `ghi_normalizado`, lags, médias móveis, `data_alvo`, `ghi_alvo`, `ghi_alvo_quantizado` e `ghi_alvo_original`.

## 15.3 Arquivos oficiais de resultados

Fluxo diário:

```text
resultados/todas_localidades/metricas_geral.csv
resultados/todas_localidades/resumo_localidades.csv
resultados/todas_localidades/estatisticas_horarias.csv
resultados/todas_localidades/previsoes/<localidade>/previsoes_modelos.csv
resultados/todas_localidades/previsoes/<localidade>/previsoes_xgboost.csv
resultados/todas_localidades/previsoes/<localidade>/previsoes_mlp.csv
resultados/todas_localidades/previsoes/<localidade>/previsoes_rnn.csv
resultados/todas_localidades/previsoes/<localidade>/previsoes_lstm.csv
```

Fluxo mensal:

```text
resultados/todas_localidades_mensal/metricas_geral.csv
resultados/todas_localidades_mensal/resumo_localidades.csv
resultados/todas_localidades_mensal/estatisticas_horarias.csv
resultados/todas_localidades_mensal/previsoes/<localidade>/previsoes_modelos.csv
```

## 15.4 Modelos salvos

```text
resultados/modelos/localidades/*.joblib
resultados/modelos/localidades/*.keras
resultados/modelos/localidades_mensal/*.joblib
resultados/modelos/localidades_mensal/*.keras
```

`joblib` é usado para XGBoost e MLP. O formato `.keras` é usado para RNN e LSTM.

## 15.5 Experimentos avançados

```text
resultados/experimentos_redes_avancadas/status_execucao.csv
resultados/experimentos_redes_avancadas/metricas_experimentos_todas.csv
resultados/experimentos_redes_avancadas/diaria/metricas_experimentos.csv
resultados/experimentos_redes_avancadas/mensal/metricas_experimentos.csv
resultados/experimentos_redes_avancadas/diaria/comparacao_com_modelos_oficiais.csv
resultados/experimentos_redes_avancadas/mensal/comparacao_com_modelos_oficiais.csv
```

Esses arquivos são lidos pelo notebook `03_experimentos_redes_avancadas.ipynb`.


# 16. Resultados atuais

Os resultados oficiais já estão salvos em CSV e são apresentados com mais detalhes no notebook `02_resultados_todas_localidades.ipynb`.

## 16.1 Resultado diário

O arquivo principal é:

```text
resultados/todas_localidades/metricas_geral.csv
```

Ele possui uma linha por par localidade/modelo. Como são 10 localidades e 4 modelos oficiais, o total esperado é:

```text
10 localidades × 4 modelos = 40 linhas
```

O resumo por localidade fica em:

```text
resultados/todas_localidades/resumo_localidades.csv
```

A coluna `Melhor_Modelo` indica o vencedor pelo maior `R2_wm2`.

## 16.2 Resultado mensal

O fluxo mensal usa os mesmos modelos, mas com série agregada por mês:

```text
resultados/todas_localidades_mensal/metricas_geral.csv
resultados/todas_localidades_mensal/resumo_localidades.csv
```

Esse fluxo também gera 40 linhas de métricas, pois avalia os mesmos 4 modelos nas mesmas 10 localidades.

## 16.3 Como ler os resultados

- `MAE_wm2` e `RMSE_wm2` mostram o erro na unidade física aproximada;
- `nRMSE_percentual_wm2` mostra o erro relativo à média da localidade;
- `R2_wm2` indica poder explicativo na escala física;
- `Melhor_Modelo` usa o maior `R2_wm2`, não o menor MAE;
- não existe vencedor universal garantido para todas as localidades.

## 16.4 Notebooks de apresentação

```text
01_explicacao_teorica_pipeline.ipynb      # metodologia detalhada
02_resultados_todas_localidades.ipynb     # resultados oficiais e feedbacks resumidos
03_experimentos_redes_avancadas.ipynb     # resultados dos modelos experimentais
```


# 17. Mapa dos arquivos Python

Esta tabela mostra onde cada parte realmente acontece. Ela é importante porque os notebooks apresentam e explicam, mas os cálculos oficiais estão nos arquivos `.py`.

| Arquivo | Responsabilidade | Principais resultados ou efeitos |
|---|---|---|
| `codigo_fonte/configuracao.py` | centraliza caminhos do projeto | define pastas `dados/`, `resultados/`, `modelos/`, `relatorios/` |
| `codigo_fonte/localidades_ev.py` | cadastro auditável das 10 fábricas | nomes, países, coordenadas, fontes e distância Haversine |
| `codigo_fonte/preprocessamento.py` | coleta, leitura, limpeza, média diária/mensal, quantização e normalização | devolve `PreparationResult` e salva arquivos `*_features.csv` |
| `codigo_fonte/features.py` | cria lags, médias móveis, alvo `t+1` e divisão temporal | gera `X_train`, `X_test`, `y_train`, `y_test` |
| `codigo_fonte/modelos.py` | treina XGBoost, MLP, RNN e LSTM | salva modelos `.joblib` e `.keras` |
| `codigo_fonte/avaliacao.py` | desnormaliza previsões, calcula métricas e salva CSVs | gera métricas e `previsoes_modelos.csv` |
| `codigo_fonte/graficos.py` | gera gráficos de série temporal e dispersão | salva figuras PNG em `resultados/.../figuras/` |
| `treinamento_principal.py` | executa o pipeline para uma única série | resultados em `resultados/metricas/`, `resultados/modelos/` e `resultados/figuras/` |
| `treinar_todas_localidades.py` | valida dados e treina as 10 localidades | gera os resultados oficiais diários e mensais |
| `experimentos_redes_avancadas.py` | roda DilatedRNN, DeepAR_exp e DeepNPTS_aprox | gera resultados em `resultados/experimentos_redes_avancadas/` |

## 17.1 Arquivo mais importante para os resultados oficiais

O arquivo que de fato consolida os resultados das dez localidades é:

```text
treinar_todas_localidades.py
```

Ele faz quatro coisas principais:

1. valida ou coleta os CSVs oficiais;
2. chama `preparar_serie_temporal` para cada localidade;
3. treina XGBoost, MLP, RNN e LSTM;
4. salva métricas, previsões, modelos e gráficos.

## 17.2 Funções principais no fluxo oficial

```text
treinar_todas_localidades.py
    ├── validar_csv_nrel_localidade
    ├── carregar_ou_coletar_localidade
    ├── treinar_localidade
    ├── consolidar_resultados
    └── executar_lote_modelagem
```

Dentro de `treinar_localidade`, o fluxo chama:

```text
preparar_serie_temporal        -> codigo_fonte/preprocessamento.py
dividir_treino_teste_temporal  -> codigo_fonte/features.py
treinar_xgboost                -> codigo_fonte/modelos.py
treinar_mlp                    -> codigo_fonte/modelos.py
treinar_rnn                    -> codigo_fonte/modelos.py
treinar_lstm                   -> codigo_fonte/modelos.py
calcular_metricas              -> codigo_fonte/avaliacao.py
salvar_previsoes               -> codigo_fonte/avaliacao.py
salvar_graficos                -> codigo_fonte/graficos.py
```

## 17.3 Onde estão os resultados finais

| Resultado | Caminho |
|---|---|
| métricas diárias oficiais | `resultados/todas_localidades/metricas_geral.csv` |
| resumo diário por localidade | `resultados/todas_localidades/resumo_localidades.csv` |
| previsões diárias linha a linha | `resultados/todas_localidades/previsoes/<localidade>/previsoes_modelos.csv` |
| métricas mensais oficiais | `resultados/todas_localidades_mensal/metricas_geral.csv` |
| resumo mensal por localidade | `resultados/todas_localidades_mensal/resumo_localidades.csv` |
| previsões mensais linha a linha | `resultados/todas_localidades_mensal/previsoes/<localidade>/previsoes_modelos.csv` |
| modelos diários treinados | `resultados/modelos/localidades/` |
| modelos mensais treinados | `resultados/modelos/localidades_mensal/` |
| métricas experimentais | `resultados/experimentos_redes_avancadas/metricas_experimentos_todas.csv` |

## 17.4 Papel dos notebooks

| Notebook | Papel |
|---|---|
| `00_coleta_dados_localidades.ipynb` | apoio para coleta e inspeção dos dados |
| `01_explicacao_teorica_pipeline.ipynb` | explicação detalhada da metodologia e do projeto |
| `02_resultados_todas_localidades.ipynb` | apresentação resumida dos resultados oficiais |
| `03_experimentos_redes_avancadas.ipynb` | apresentação dos resultados experimentais |

A regra prática é: **se a pergunta é “como foi calculado?”, procure os arquivos `.py`; se a pergunta é “como apresentar e interpretar?”, use os notebooks.**


# 18. Como executar

## Instalar dependências

```bash
pip install -r requirements.txt
```

## Validar os dados oficiais

```bash
python treinar_todas_localidades.py --validar-dados
```

## Baixar novamente os dados

Requer `NREL_API_KEY` e `NREL_EMAIL` no arquivo `.env`:

```bash
python treinar_todas_localidades.py --somente-download --forcar-download
```

## Treinar as dez localidades

```bash
python treinar_todas_localidades.py
```

## Treinar uma única série

```bash
python treinamento_principal.py \
  --data-path dados/brutos/localidades_ev/byd_camacari.csv
```

## Rodar os modelos avançados

```bash
python experimentos_redes_avancadas.py --frequencia ambas
```

Os resultados ficam em `resultados/experimentos_redes_avancadas/` e são apresentados no notebook `03_experimentos_redes_avancadas.ipynb`.

## Executar testes

```bash
pytest
```



# 19. Limitações e melhorias possíveis

## Limitações atuais

1. **Somente o histórico do GHI é usado.** Não entram nuvens, temperatura, umidade, precipitação ou previsão meteorológica.
2. **Horizonte único.** O modelo prevê apenas um dia à frente.
3. **Divisão única 80/20.** Não há validação temporal em múltiplas janelas.
4. **Hiperparâmetros fixos.** Não foi realizada busca sistemática por configuração.
5. **Quantização perde resolução.** É necessário comparar com treinamento diretamente no GHI contínuo.
6. **Média diária inclui horas noturnas.** Isso é coerente com o valor médio de 24 horas, mas outra definição poderia usar apenas período diurno ou energia integrada.
7. **Features sazonais explícitas não são usadas.** Mês, dia do ano, seno/cosseno sazonais e posição solar poderiam ajudar.
8. **Não existe baseline explícito nas tabelas.** Comparar com persistência, como `previsão = GHI de hoje`, fortaleceria a avaliação.
9. **COV horário depende da série horária original.** A API fornece dados de 60 minutos, mas os CSVs salvos atualmente já estão agregados por média diária. Depois dessa agregação, não é possível reconstruir `sigma/média` horário com rigor.
10. **Modelos são locais.** Não existe um modelo global que aprenda conjuntamente com latitude, longitude e localidade.

## Melhorias prioritárias

- adicionar baseline de persistência;
- incluir variáveis meteorológicas;
- usar validação *walk-forward*;
- comparar série contínua contra série quantizada;
- adicionar features sazonais;
- otimizar hiperparâmetros somente dentro do treino;
- salvar parâmetros de transformação junto ao modelo;
- recoletar os CSVs horários quando necessário para preencher o COV horário das bases antigas.

Esses pontos não invalidam o trabalho. Eles delimitam o que foi avaliado e indicam continuidade científica.


# 20. Perguntas comuns de banca

## “A média usada é diária ou mensal?”

Existem dois fluxos. O fluxo diário usa a média das observações horárias de cada dia e prevê o dia seguinte. O fluxo mensal agrega os valores diários por mês civil e prevê o mês seguinte.

## “Por que a unidade continua W/m²?”

Porque foi calculada uma média temporal da irradiância, não uma integral de energia. Uma média de valores em `W/m²` continua em `W/m²`.

## “O modelo vê o valor do dia que deve prever?”

Não. A linha do período `t` usa dados até `t` e prevê `t+1`. Esse alinhamento é feito com `shift(-1)` para o alvo e com janelas que terminam em `t` para as features.

## “Por que não embaralhar os dados?”

Porque a aplicação real usa passado para prever futuro. Embaralhar poderia colocar dados futuros no treino e produzir uma avaliação otimista.

## “Onde estão de fato os cálculos dos modelos?”

Os modelos ficam em `codigo_fonte/modelos.py`. O treinamento em lote das dez localidades fica em `treinar_todas_localidades.py`. As métricas ficam em `codigo_fonte/avaliacao.py`.

## “Onde estão de fato os resultados?”

Os resultados oficiais ficam em:

```text
resultados/todas_localidades/metricas_geral.csv
resultados/todas_localidades/resumo_localidades.csv
resultados/todas_localidades_mensal/metricas_geral.csv
resultados/todas_localidades_mensal/resumo_localidades.csv
```

As previsões linha a linha ficam nas pastas `previsoes/<localidade>/`.

## “Por que RNN/LSTM se as features são tabulares?”

Elas foram incluídas para comparar redes recorrentes com modelos tabulares. As sete features temporais são reinterpretadas como uma sequência curta de sete passos. Isso não transforma a entrada em uma série horária completa, mas permite testar uma leitura recorrente das defasagens e médias móveis.

## “As métricas estão normalizadas ou em W/m²?”

As duas versões são salvas. As métricas normalizadas avaliam a escala interna do modelo. As métricas em `W/m²` facilitam a interpretação física do erro.

## “Os modelos avançados entram no resultado principal?”

Não automaticamente. DilatedRNN, DeepAR_exp e DeepNPTS_aprox foram executados como rodada complementar em `experimentos_redes_avancadas.py`. Eles são úteis para comparação, mas ficam separados do pipeline oficial.


# 21. Roteiro curto para apresentar o projeto

> “O trabalho avalia a previsão diária e mensal de irradiância solar em dez localidades de fábricas de veículos elétricos. Os dados vêm da base oficial NLR/NSRDB, originalmente com intervalo de 60 minutos, e são agregados em uma média para cada dia entre 2019 e 2024. Em seguida, um fluxo complementar agrega esses valores por mês civil.
>
> Depois da limpeza, o GHI é quantizado em 128 níveis e normalizado entre zero e um. Para transformar a série temporal em um problema supervisionado, criamos defasagens e médias móveis. Cada linha usa informações disponíveis até o período atual para prever o próximo período.
>
> A separação é cronológica: os primeiros 80% dos exemplos são usados no treinamento e os 20% finais no teste. Comparamos XGBoost, MLP, RNN e LSTM usando MAE, MSE, RMSE, R² e nRMSE. As métricas foram mantidas na escala normalizada e também recalculadas após desnormalizar o GHI para `W/m²`, o que facilita interpretar o tamanho físico dos erros.
>
> Além disso, o pipeline registra o COV horário, calculado como `sigma/média` do GHI horário antes da agregação diária quando a série horária original está disponível. Também foi realizada uma rodada complementar com DilatedRNN, DeepAR experimental e DeepNPTS aproximado, mantendo o mesmo corte temporal e as mesmas métricas. Os resultados mostram que não existe um único vencedor garantido em todas as localidades.”


# 22. Resumo final

| Pergunta | Resposta |
|---|---|
| O que é previsto? | GHI médio diário do dia seguinte e GHI médio mensal do mês seguinte |
| Qual é a fonte? | NLR/NSRDB, GOES Aggregated PSM v4 |
| Qual período? | 2019–2024 |
| Qual granularidade do modelo? | diária e mensal |
| A média mensal entra no modelo? | sim, no fluxo mensal complementar |
| Quais features? | diário: lags 1, 2, 3 e 7; médias móveis 3, 7 e 30 dias. mensal: lags 1, 2, 3 e 6; médias móveis 3, 6 e 12 meses |
| Qual alvo? | GHI quantizado e normalizado de `t+1`, onde `t+1` é o próximo dia ou o próximo mês |
| Quantos exemplos por local? | diário: 2.162; mensal: 60 após remover linhas sem histórico completo e sem alvo futuro |
| Como é a divisão? | 80% treino e 20% teste, em ordem temporal |
| Quais modelos oficiais? | XGBoost, MLP, RNN e LSTM |
| Quais modelos experimentais? | DilatedRNN, DeepAR_exp e DeepNPTS_aprox |
| Quais métricas? | MAE, MSE, RMSE, R² e nRMSE, nas escalas normalizada e `W/m²` |
| O que é COV horário? | `sigma/média` do GHI horário, calculado antes da média diária |
| Como se evita vazamento? | escala no treino, features até `t`, alvo em `t+1` e divisão cronológica |

O ponto mais importante é compreender que o projeto mantém a causalidade temporal: **o passado e o presente formam as entradas; o próximo período é o alvo**. Para apresentação dos resultados, a escala em `W/m²` deve acompanhar a escala normalizada porque ela mostra a magnitude física do erro.
